In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Literal
from typing_extensions import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from IPython.display import Image, display

# Load env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    api_key=api_key
)

# -----------------------------
# Graph State
# -----------------------------
class State(TypedDict):
    joke: str
    topic: str
    feedback: str
    funny_or_not: str

# -----------------------------
# Schema for evaluator
# -----------------------------
class Feedback(BaseModel):
    grade: Literal["funny", "not funny"] = Field(
        description="Decide if the joke is funny or not."
    )
    feedback: str = Field(
        description="If the joke is not funny, give feedback on improving it."
    )

# Augmented LLM for structured output
evaluator = llm.with_structured_output(Feedback)

# -----------------------------
# Nodes
# -----------------------------
def llm_call_generator(state: State):
    """Generate a joke, consider feedback if present"""
    if state.get("feedback"):
        prompt = f"Write a better joke about {state['topic']}. Improve using this feedback: {state['feedback']}"
    else:
        prompt = f"Write a joke about {state['topic']}"

    msg = llm.invoke(prompt)
    return {"joke": msg.content}


def llm_call_evaluator(state: State):
    """Evaluate the joke"""
    grade = evaluator.invoke(f"Grade this joke: {state['joke']}")
    return {
        "funny_or_not": grade.grade,
        "feedback": grade.feedback
    }

# -----------------------------
# Conditional routing
# -----------------------------
def route_joke(state: State):
    if state["funny_or_not"] == "funny":
        return "Accepted"
    else:
        return "Rejected + Feedback"

# -----------------------------
# Build workflow
# -----------------------------
optimizer_builder = StateGraph(State)

optimizer_builder.add_node("llm_call_generator", llm_call_generator)
optimizer_builder.add_node("llm_call_evaluator", llm_call_evaluator)

optimizer_builder.add_edge(START, "llm_call_generator")
optimizer_builder.add_edge("llm_call_generator", "llm_call_evaluator")

optimizer_builder.add_conditional_edges(
    "llm_call_evaluator",
    route_joke,
    {
        "Accepted": END,
        "Rejected + Feedback": "llm_call_generator",
    },
)

optimizer_workflow = optimizer_builder.compile()

# Visualize
display(Image(optimizer_workflow.get_graph().draw_mermaid_png()))

# Run workflow
state = optimizer_workflow.invoke({"topic": "Cats"})
print(state["joke"])


ModuleNotFoundError: No module named 'dotenv'

: 

In [1]:
%pip install -U "langgraph-cli[inmem]"


  Using cached langgraph_api-0.5.14-py3-none-any.whl.metadata (4.2 kB)
  Using cached langgraph_runtime_inmem-0.18.0-py3-none-any.whl.metadata (570 bytes)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached cryptography-44.0.3-cp39-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached grpcio_tools-1.75.1-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached grpcio-1.76.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached jsonschema_rs-0.29.1.tar.gz (1.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated pac

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\sam\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\sam\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\sam\.rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to 'minimal'
      info: default host triple is x86_64-pc-windows-msvc
   